# Long-Only Backtesting Class

某些投资者的偏好或监管规定可能禁止卖空。因此只能建立多头，或把资金停在现金/低风险资产（如货币市场账户）里。书中 "Long-Only Backtesting Class"（约第 194 页）给出纯多头回测类 `BacktestLongOnly`：它继承 `BacktestBase`，所以 SMA、动量、均值回归三种策略的实现都比较短。

三种策略里，均值回归的事件循环稍绕一点（空仓时跌破 SMA−threshold 才买，回到 SMA 才平）；SMA / 动量是同一套仓位状态机。完整类与运行脚本见下文。


In [1]:
import os
proxy = 'http://127.0.0.1:7897'
os.environ['HTTP_PROXY'] = proxy
os.environ['HTTPS_PROXY'] = proxy

## 1. 完整类：`BacktestLongOnly`

下面是书中用于纯多头事件驱动回测的完整类（SMA / 动量 / 均值回归）。基类从同目录的 `BacktestBase.py` 导入（由 `BacktestBase.ipynb` 抽出，便于 `import`）。


In [6]:
#
# Python Script with Long Only Class
# for Event-Based Backtesting
#
# Python for Algorithmic Trading
# (c) Dr. Yves J. Hilpisch
# The Python Quants GmbH
#
from BacktestBase import BacktestBase


class BacktestLongOnly(BacktestBase):
    '''纯多头事件驱动回测。

    仓位规则（三个策略共用）：空仓时全款买入；持仓期间信号仍成立则不动；
    信号翻空则卖光全部 units，回到空仓。不做分批加仓/减仓。
    '''

    def run_sma_strategy(self, SMA1, SMA2):
        '''回测双均线（SMA）策略。

        参数
        ====
        SMA1, SMA2: int
            短均线、长均线窗口（交易日）
        '''
        msg = f'\n\nRunning 双均线（SMA）策略 | SMA1={SMA1} & SMA2={SMA2}'
        msg += f'\nfixed costs {self.ftc} | '
        msg += f'proportional costs {self.ptc}'
        print(msg)
        print('=' * 55)
        self.position = 0  # 初始空仓
        self.trades = 0  # 尚未成交
        self.amount = self.initial_amount  # 重置为初始资金
        self.data['SMA1'] = self.data['price'].rolling(SMA1).mean()
        self.data['SMA2'] = self.data['price'].rolling(SMA2).mean()

        for bar in range(SMA2, len(self.data)):
            if self.position == 0:
                if self.data['SMA1'].iloc[bar] > self.data['SMA2'].iloc[bar]:
                    self.place_buy_order(bar, amount=self.amount)
                    self.position = 1  # 转入多头
            elif self.position == 1:
                if self.data['SMA1'].iloc[bar] < self.data['SMA2'].iloc[bar]:
                    self.place_sell_order(bar, units=self.units)
                    self.position = 0  # 回到空仓
        self.close_out(bar)

    def run_momentum_strategy(self, momentum):
        '''回测动量策略。

        参数
        ====
        momentum: int
            计算均值收益所用的交易日窗口
        '''
        msg = f'\n\nRunning 动量策略 | {momentum} days'
        msg += f'\nfixed costs {self.ftc} | '
        msg += f'proportional costs {self.ptc}'
        print(msg)
        print('=' * 55)
        self.position = 0  # 初始空仓
        self.trades = 0  # 尚未成交
        self.amount = self.initial_amount  # 重置为初始资金
        self.data['momentum'] = self.data['return'].rolling(momentum).mean()
        for bar in range(momentum, len(self.data)):
            if self.position == 0:
                if self.data['momentum'].iloc[bar] > 0:
                    self.place_buy_order(bar, amount=self.amount)
                    self.position = 1  # 转入多头
            elif self.position == 1:
                if self.data['momentum'].iloc[bar] < 0:
                    self.place_sell_order(bar, units=self.units)
                    self.position = 0  # 回到空仓
        self.close_out(bar)

    def run_mean_reversion_strategy(self, SMA, threshold):
        '''回测均值回归策略。

        参数
        ====
        SMA: int
            均线窗口（交易日）
        threshold: float
            相对 SMA 的绝对偏离阈值（价格单位）
        '''
        msg = f'\n\nRunning 均线窗口策略 | '
        msg += f'SMA={SMA} & thr={threshold}'
        msg += f'\nfixed costs {self.ftc} | '
        msg += f'proportional costs {self.ptc}'
        print(msg)
        print('=' * 55)
        self.position = 0  # 初始空仓
        self.trades = 0  # 尚未成交
        self.amount = self.initial_amount  # 重置为初始资金
        self.data['SMA'] = self.data['price'].rolling(SMA).mean()
        for bar in range(SMA, len(self.data)):
            if self.position == 0:
                if (self.data['price'].iloc[bar] < self.data['SMA'].iloc[bar] - threshold):
                    self.place_buy_order(bar, amount=self.amount)
                    self.position = 1  # 转入多头
            elif self.position == 1:
                if self.data['price'].iloc[bar] >= self.data['SMA'].iloc[bar]:
                    self.place_sell_order(bar, units=self.units)
                    self.position = 0  # 回到空仓
        self.close_out(bar)


## 2. 运行

先在无交易成本下跑三个策略，再在固定成本 10 USD、比例成本 1% 下重跑。需要能访问 `hilpisch.com` 的样例 CSV。


In [7]:
def run_strategies(lobt):
    lobt.run_sma_strategy(SMA1=42, SMA2=252)
    lobt.run_momentum_strategy(momentum=60)
    lobt.run_mean_reversion_strategy(SMA=50, threshold=5)


lobt = BacktestLongOnly(
    symbol='AAPL.O',
    start='2010-1-1',
    end='2019-12-31',
    amount=10000,
    verbose=False,
)
run_strategies(lobt)



Running 双均线（SMA）策略 | SMA1=42 & SMA2=252
fixed costs 0.0 | proportional costs 0.0
Final balance   [$] 56204.95
Net Performance [%] 462.05
Trades Executed [#] 10.00


Running 动量策略 | 60 days
fixed costs 0.0 | proportional costs 0.0
Final balance   [$] 82969.30
Net Performance [%] 729.69
Trades Executed [#] 122.00


Running 均线窗口策略 | SMA=50 & thr=5
fixed costs 0.0 | proportional costs 0.0
Final balance   [$] 11543.77
Net Performance [%] 15.44
Trades Executed [#] 41.00


In [8]:

# 交易成本：固定 10 USD + 比例 1%
lobt = BacktestLongOnly(
    symbol='AAPL.O',
    start='2010-1-1',
    end='2019-12-31',
    amount=10000,
    ftc=10.0,
    ptc=0.01,
    verbose=False,
)
run_strategies(lobt)




Running 双均线（SMA）策略 | SMA1=42 & SMA2=252
fixed costs 10.0 | proportional costs 0.01
Final balance   [$] 51959.62
Net Performance [%] 419.60
Trades Executed [#] 10.00


Running 动量策略 | 60 days
fixed costs 10.0 | proportional costs 0.01
Final balance   [$] 22907.27
Net Performance [%] 129.07
Trades Executed [#] 122.00


Running 均线窗口策略 | SMA=50 & thr=5
fixed costs 10.0 | proportional costs 0.01
Final balance   [$] 7400.70
Net Performance [%] -25.99
Trades Executed [#] 41.00


上面脚本会在输出区打印三组策略结果。通过对比无交易成本和有交易成本的三个策略基于事件的回测结果，可以得出：
- 交易成本会侵蚀业绩；
- 交易越频，拖累越大。
- 无成本时动量往往强于 SMA，加上成本后交易更少的 SMA 反而可能反超。


## 3. 结果分析

有无交易成本下的成交笔数与净收益如下（初始资金 10000 USD；有成本为固定 10 USD、比例 1%）。成交笔数不随成本变化；差异来自信号构造，而非实现错误。

| 策略 | 参数 | 成交笔数 | 无成本净收益 | 有成本净收益 |
| --- | --- | ---: | ---: | ---: |
| 双均线（SMA） | SMA1=42, SMA2=252 | 10 | 462.05% | 419.60% |
| 动量 | 60 日 | 122 | 729.69% | 129.07% |
| 均线窗口 | SMA=50, threshold=5 | 41 | 15.44% | -25.99% |

动量的信号是最近 60 个交易日收益率均值的过零：大于 0 全仓买入，小于 0 清仓。该统计量围绕 0 波动较频繁。即便中长期趋势向上，一次回调即可使窗口均值转负并触发平仓，随后再转正又全仓买入。因此十年样本出现约 60 次完整进出（122 笔）是规则的自然结果。相对 SMA，它比较的是短期收益符号，而不是两条慢速均线的相对位置，故更敏感。

SMA 的信号是 42 日均线与 252 日均线交叉。252 日接近一年，交叉间隔通常以月计。交叉之后两条均线可在同一侧维持较长时间，持仓期内不再下单。扣除预热后的有效样本约八年，完整进出大约五次，对应 10 笔成交。

均线窗口是均值回归：价格跌破 SMA50 减 5 才买入，价格回到 SMA50 即卖出。2010–2019 年 AAPL 为趋势市，收益主要来自持有上涨；该规则在价格回到均线时离场，后续趋势无法计入，空仓阶段还在等待下一次跌破。`threshold=5` 是绝对价格而非百分比：低价区 5 美元偏离很大、触发稀少；高价区 5 美元接近噪声，进出更碎，也更容易买在下跌中继。纯多头只能低买、均线卖出，涨过头一侧的做空收益被排除。因此问题不在窗口微调，而在策略类型与这段行情相反：SMA 与动量顺势，均线窗口逆势且提前离场；再叠加 41 笔交易成本，无成本仅余约 15%，有成本则为亏损。


上一节我们强调，业绩需要看两个方面：方向预测的命中率，以及市场择时。本节跑完后对照输出可以看出还有第三面：策略触发交易的次数。

高频策略更容易被成本吃掉所谓超额表现，这也常被用来支持低成本被动策略（例如 ETF）。
